In [ ]:
FW_DIR = "../fw/examples/simpleserial_example"
BIN_FILE = f"{FW_DIR}/simpleserial_example.bin"
BOOTLOADER = "python3 ../../sdk/toolchain/bootloader.py"
BS_FILE = "../vivado/risqrypt_cw305.runs/impl_1/fpga_top.bit"
TARGET_PLATFORM = 'CW305_100t'

In [ ]:
import sys
sys.path.insert(0, "../")

from host.CW305_rq import CW305_rq

In [ ]:
import chipwhisperer as cw
scope = cw.scope()
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled"

In [ ]:
target = cw.target(scope, CW305_rq, force=True, fpga_id='100t', platform='cw305', bsfile=BS_FILE)

In [ ]:
target.vccint_set(1.0)
# we only need PLL1:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)

# run at 10 MHz:
target.pll.pll_outfreq_set(10E6, 1)

# 1ms is plenty of idling time
target.clkusbautooff = True
target.clksleeptime = 1

In [ ]:
%%bash -s "$FW_DIR"
cd $1
make

In [ ]:
%%bash -s "$BIN_FILE" "$BOOTLOADER"
$2 -f $1 -q

In [ ]:
if scope._is_husky:
    scope.clock.clkgen_freq = 40e6
    scope.clock.clkgen_src = 'extclk'
    scope.clock.adc_mul = 4
    # if the target PLL frequency is changed, the above must also be changed accordingly
else:
    scope.clock.adc_src = "extclk_x4"

In [ ]:
import time
for i in range(5):
    scope.clock.reset_adc()
    time.sleep(1)
    if scope.clock.adc_locked:
        break 
assert (scope.clock.adc_locked), "ADC failed to lock"

In [ ]:
scope.adc.samples = 5000
scope.gain.db = 20

In [ ]:
target.output_len = 16

In [ ]:
from tqdm.notebook import tnrange
import numpy as np
import estraces
import time

ktp = cw.ktp.Basic()

N = 1000  # Number of traces

# initialize cipher to verify DUT result:
key, text = ktp.next()
target.simpleserial_write('k', key, end='\n')
target.simpleserial_wait_ack()
# target.set_key(key, always_send=True, ack=True)

traces = []
textin = []
keys = []

for i in tnrange(N, desc='Capturing traces'):
    # run aux stuff that should come before trace here

    key, text = ktp.next()  # manual creation of a key, text pair can be substituted here
    textin.append(text)
    keys.append(key)
    
    ret = cw.capture_trace(scope, target, text, None)
    if not ret:
        print("Failed capture")
        continue

    textout_cw = cw.bytearray(ret.textout)
    for i in range(16):
        assert textout_cw[i] == ((key[i] + text[i]) % 256), f"mismatch at index {i}"
        
    traces.append(ret.wave)


ths = estraces.read_ths_from_ram(np.array(traces), k=np.array(keys), p=np.array(textin))

print(ths)

In [ ]:
print(ths)

In [ ]:
from matplotlib import pyplot as plt

%matplotlib widget

fig, ax = plt.subplots()

mean_trace = np.mean(np.array(traces), axis=0)
ax.plot(mean_trace)

In [ ]:

fig, ax = plt.subplots()

for i in range(5):
    ax.plot(traces[i])

In [ ]:
import scared

@scared.reverse_selection_function
def select(k, p):
    return k + p


cpa_reverse = scared.CPAReverse(selection_function=select, model=scared.HammingWeight(expected_dtype='uint8'))


container = scared.Container(ths)
cpa_reverse.run(container)


In [ ]:
cpa_reverse.results.shape

In [ ]:
fig, ax = plt.subplots()

for i in range(16):
    ax.plot(cpa_reverse.results[i])